In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

MED_PATH = "/content/drive/MyDrive/HI-Medium_Trans.csv"
usecols = ["From Bank","Account","To Bank","Account.1","Is Laundering"]

med = pd.read_csv(MED_PATH, usecols=usecols)
print("HI-Medium rows:", len(med), "pos rate:", med["Is Laundering"].mean())

sep = "\x1f"  # safe separator

src_key = med["From Bank"].astype(str) + sep + med["Account"].astype(str)
dst_key = med["To Bank"].astype(str)   + sep + med["Account.1"].astype(str)

all_keys = pd.concat([src_key, dst_key], ignore_index=True)  # 1D Series
node_ids, uniques = pd.factorize(all_keys, sort=True)

num_nodes = len(uniques)
print("Medium num_nodes:", num_nodes)

n = len(med)
src_md = node_ids[:n].astype(np.int64)
dst_md = node_ids[n:].astype(np.int64)
y_md   = med["Is Laundering"].astype(np.float32).to_numpy()

print("Edges:", len(y_md), "pos rate:", float(y_md.mean()))
print("src range:", (src_md.min(), src_md.max()), "dst range:", (dst_md.min(), dst_md.max()))

HI-Medium rows: 31898238 pos rate: 0.0011044497191349566
Medium num_nodes: 2077023
Edges: 31898238 pos rate: 0.0011044497368857265
src range: (np.int64(0), np.int64(2077022)) dst range: (np.int64(0), np.int64(2077022))


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from tqdm import trange
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- use MEDIUM fresh-start arrays ---
src = src_md
dst = dst_md
y   = y_md
n_nodes = num_nodes

# --- split indices ---
N = len(y)
rng = np.random.default_rng(0)
perm = rng.permutation(N)

val_frac = 0.1
n_val = int(N * val_frac)
val_idx_np = perm[:n_val]
tr_idx_np  = perm[n_val:]

print("train edges:", len(tr_idx_np), "val edges:", len(val_idx_np),
      "train pos:", float(y[tr_idx_np].mean()), "val pos:", float(y[val_idx_np].mean()))

# --- tensors ---
src_t = torch.from_numpy(src).long()
dst_t = torch.from_numpy(dst).long()
y_t   = torch.from_numpy(y).float()

# (optional) pinning helps host->gpu copies, but uses extra RAM
# src_t = src_t.pin_memory(); dst_t = dst_t.pin_memory(); y_t = y_t.pin_memory()

src_gpu = src_t.to(device, non_blocking=True)
dst_gpu = dst_t.to(device, non_blocking=True)
y_gpu   = y_t.to(device, non_blocking=True)

tr_idx  = torch.from_numpy(tr_idx_np).long()   # keep on CPU
val_idx = torch.from_numpy(val_idx_np).long()  # keep on CPU

# --- model ---
class EdgeMLP(nn.Module):
    def __init__(self, n_nodes, d=128, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, d)
        self.drop = nn.Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(4*d, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, s, t):
        Hs = self.drop(self.emb(s))
        Hd = self.drop(self.emb(t))
        x = torch.cat([Hs, Hd, (Hs - Hd).abs(), Hs * Hd], dim=1)
        return self.net(x).squeeze(-1)

model = EdgeMLP(n_nodes, d=128, dropout=0.2).to(device)

# --- pos_weight from TRAIN ONLY ---
pos_rate = float(y[tr_idx_np].mean())
pos_weight = torch.tensor([(1.0 - pos_rate) / max(pos_rate, 1e-12)], device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

BATCH = 262144
EPOCHS = 5  # increase if val improves

@torch.no_grad()
def eval_on(idx_t, max_n=500_000):
    model.eval()
    # optional cap for speed
    if idx_t.numel() > max_n:
        idx_t = idx_t[torch.randperm(idx_t.numel())[:max_n]]

    ps, ys = [], []
    for i in range(0, idx_t.numel(), BATCH):
        batch = idx_t[i:i+BATCH].to(device, non_blocking=True)
        p = torch.sigmoid(model(src_gpu[batch], dst_gpu[batch])).detach().cpu().numpy()
        yt = y_gpu[batch].detach().cpu().numpy()
        ps.append(p); ys.append(yt)

    ps = np.concatenate(ps); ys = np.concatenate(ys)
    return (
        roc_auc_score(ys, ps),
        average_precision_score(ys, ps),
        float(ps.mean()),
        float(ys.mean()),
    )

# --- training ---
for ep in range(EPOCHS):
    model.train()
    order = tr_idx[torch.randperm(tr_idx.numel())]  # CPU shuffle

    running = 0.0
    for i in trange(0, order.numel(), BATCH, desc=f"train ep{ep}"):
        batch = order[i:i+BATCH].to(device, non_blocking=True)
        s, t, yy = src_gpu[batch], dst_gpu[batch], y_gpu[batch]

        opt.zero_grad(set_to_none=True)
        logits = model(s, t)
        loss = loss_fn(logits, yy)
        loss.backward()
        opt.step()

        running += loss.detach().item() * s.size(0)

    tr_roc, tr_pr, tr_pmean, tr_ymean = eval_on(tr_idx, max_n=500_000)
    va_roc, va_pr, va_pmean, va_ymean = eval_on(val_idx, max_n=500_000)

    print(
        f"Epoch {ep:02d} | loss={running/order.numel():.6f} "
        f"| TRAIN ROC={tr_roc:.4f} PR={tr_pr:.4f} "
        f"| VAL ROC={va_roc:.4f} PR={va_pr:.4f} "
        f"| val_pred_mean={va_pmean:.6f} val_label_mean={va_ymean:.6f}"
    )

device: cuda
train edges: 28708415 val edges: 3189823 train pos: 0.0011097791139036417 val pos: 0.0010564849944785237


train ep0: 100%|██████████| 110/110 [00:22<00:00,  4.98it/s]


Epoch 00 | loss=1.339550 | TRAIN ROC=0.6232 PR=0.0018 | VAL ROC=0.5829 PR=0.0015 | val_pred_mean=0.492245 val_label_mean=0.001096


train ep1: 100%|██████████| 110/110 [00:21<00:00,  5.16it/s]


Epoch 01 | loss=1.322518 | TRAIN ROC=0.6372 PR=0.0132 | VAL ROC=0.6100 PR=0.0059 | val_pred_mean=0.475980 val_label_mean=0.001058


...

Epoch 02 | loss=1.292578 | TRAIN ROC=0.7161 PR=0.0221 | VAL ROC=0.6487 PR=0.0092 | val_pred_mean=0.471671 val_label_mean=0.001062


train ep3: 100%|██████████| 110/110 [00:21<00:00,  5.14it/s]


Epoch 03 | loss=1.226488 | TRAIN ROC=0.8051 PR=0.0694 | VAL ROC=0.6896 PR=0.0439 | val_pred_mean=0.417215 val_label_mean=0.001102


train ep4: 100%|██████████| 110/110 [00:21<00:00,  5.13it/s]


Epoch 04 | loss=1.112151 | TRAIN ROC=0.8991 PR=0.1221 | VAL ROC=0.7240 PR=0.0793 | val_pred_mean=0.406468 val_label_mean=0.001056
